In [1]:
PATH_WORK_DIR = ".."

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.chdir(PATH_WORK_DIR)
print(f"DIRECTORY: {os.getcwd()}")

DIRECTORY: c:\Users\jayar\Desktop\바탕 화면\REPO\PROJECT\M2-PJT_STATS


# package

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# data

In [6]:
LOAD_DIR = "./data/origin"
FILES = list(Path(LOAD_DIR).glob("*.csv"))

dfs = [
    pd.read_csv(
        filepath_or_buffer=file, 
        encoding="cp949",
        skiprows=15,
    ) 
    for file in FILES
]

df = pd.concat(dfs, ignore_index=True)

In [7]:
origin = df.copy()

# preprocessing

In [8]:
df['시'] = df['시군구'].str.split().str[0]
df['구'] = df['시군구'].str.split().str[1]

COND = df["구"]=="과천시"
df.loc[COND, "시"] = "서울특별시"

df["REGION"] = df["시"].astype(str) + "_" + df["구"].astype(str)

In [9]:
df["YEARMONTH"] = pd.to_datetime(
    df["계약년월"].astype(str),
    format="%Y%m",
).dt.to_period("M")

In [10]:
df["거래금액(만원)"] = df["거래금액(만원)"].str.replace(",", "")
df["단위가격"] = df["거래금액(만원)"].astype(float) / df["전용면적(㎡)"].astype(float)
df["PRICE"] = np.log(df["단위가격"])

In [11]:
df = (
    df
    .groupby(["REGION", "YEARMONTH"])["PRICE"]
    .mean()
    .reset_index()
)

In [12]:
df["PERIOD_0"] = 0
df["PERIOD_1"] = 0
df["PERIOD_2"] = 0

COND = (df["YEARMONTH"] >= "2016-01") & (df["YEARMONTH"] <= "2017-07")
df.loc[COND, "PERIOD_0"] = 1

COND = (df["YEARMONTH"] >= "2017-08") & (df["YEARMONTH"] <= "2018-08")
df.loc[COND, "PERIOD_1"] = 1

COND = (df["YEARMONTH"] >= "2018-09") & (df["YEARMONTH"] <= "2019-12")
df.loc[COND, "PERIOD_2"] = 1

In [13]:
df["TREATMENT"] = 0

TREATMENT = ["서울특별시", "세종특별자치시"]
TREATMENT = "|".join(TREATMENT)
COND = df["REGION"].str.contains(TREATMENT, na=False)

df.loc[COND, "TREATMENT"] = 1

In [14]:
df['CITY'] = df['REGION'].str.split("_").str[0]

In [15]:
df['TIME_IDX'] = pd.factorize(df['YEARMONTH'])[0]
df["REGION_IDX"] = pd.factorize(df['REGION'])[0]
df["CITY_IDX"] = pd.factorize(df['CITY'])[0]

In [16]:
COL_SORTED = [
    "REGION", "CITY", "YEARMONTH", 
    "REGION_IDX", "CITY_IDX", "TIME_IDX", "TREATMENT", "PERIOD_0", "PERIOD_1", "PERIOD_2", 
    "PRICE",
]
df = df[COL_SORTED]

In [17]:
df.head()

,REGION,CITY,YEARMONTH,REGION_IDX,CITY_IDX,TIME_IDX,TREATMENT,PERIOD_0,PERIOD_1,PERIOD_2,PRICE
0,광주광역시_광산구,광주광역시,2016-01,0,0,0,0,1,0,0,5.479704
1,광주광역시_광산구,광주광역시,2016-02,0,0,1,0,1,0,0,5.483194
2,광주광역시_광산구,광주광역시,2016-03,0,0,2,0,1,0,0,5.480802
3,광주광역시_광산구,광주광역시,2016-04,0,0,3,0,1,0,0,5.465429
4,광주광역시_광산구,광주광역시,2016-05,0,0,4,0,1,0,0,5.449539


# save

In [ ]:
PATH = './data/housing.csv'

kwargs = dict(
    path_or_buf=PATH,
    index=False,
)

df.to_csv(**kwargs)